# V5 — partially unfrozen NeuroLM/GPT-2 → ZuCo sentiment

V5 keeps the neural tokenizer and lower GPT-2 layers frozen, but adapts the final two GPT-2 blocks and the small label-verbalizer adapter to ZuCo. It is still **EEG-only**: every recording receives the same instruction, and the stimulus sentence is never shown to the model.

This is a long end-to-end run: aligned and shuffled controls × 5 folds × 3 seeds = 30 fits. Every completed fit is saved in Drive. If Colab disconnects, reconnect and run all cells again; completed fits are reused. Select a **GPU** runtime before starting.

In [ ]:
# 1) Fetch this project, install Colab-only dependencies, and pin official NeuroLM.
from pathlib import Path
import importlib.metadata as package_metadata
import os, subprocess, sys

PROJECT_URL = "https://github.com/parmisbathayan/EEGTokenizer.git"
PROJECT_ROOT = Path("/content/EEGTokenizer")
UPSTREAM_URL = "https://github.com/935963004/NeuroLM.git"
UPSTREAM_COMMIT = "0cda9876d8ce6ee07ed0c43eee5e9a6f5c24b177"
UPSTREAM_ROOT = Path("/content/NeuroLM")

def run(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)

if not PROJECT_ROOT.exists():
    run(["git", "clone", "--depth", "1", PROJECT_URL, str(PROJECT_ROOT)])
else:
    run(["git", "pull", "--ff-only"], cwd=PROJECT_ROOT)
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "neurolm/requirements-colab.txt")])
run([sys.executable, "-c", "from huggingface_hub import is_offline_mode; import transformers; print('dependency import check passed')"])
loaded_hub = sys.modules.get("huggingface_hub")
if loaded_hub is not None and not hasattr(loaded_hub, "is_offline_mode"):
    raise RuntimeError("huggingface_hub was imported before upgrade. Restart the runtime, then rerun Cell 1.")
if not (UPSTREAM_ROOT / ".git").exists():
    UPSTREAM_ROOT.mkdir(parents=True, exist_ok=True)
    run(["git", "init"], cwd=UPSTREAM_ROOT)
    run(["git", "remote", "add", "origin", UPSTREAM_URL], cwd=UPSTREAM_ROOT)
run(["git", "fetch", "--depth", "1", "origin", UPSTREAM_COMMIT], cwd=UPSTREAM_ROOT)
run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=UPSTREAM_ROOT)
os.chdir(PROJECT_ROOT / "neurolm")
critical_tests = [
    "tests.test_partial_finetune",
    "tests.test_gpt2_cache",
    "tests.test_gpt2_verbalizer",
    "tests.test_raw_cache",
    "tests.test_raw_eegnet",
    "tests.test_official_neurolm",
]
test_result = subprocess.run(
    [sys.executable, "-m", "unittest", *critical_tests, "-v"],
    cwd=Path.cwd(), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(test_result.stdout)
if test_result.returncode:
    raise RuntimeError("V5-critical tests failed; the complete output is printed above")
project_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
print("Project revision:", project_revision)
print("huggingface_hub:", package_metadata.version("huggingface_hub"))
print("transformers:", package_metadata.version("transformers"))
print("Official NeuroLM commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=UPSTREAM_ROOT, text=True).strip())

In [ ]:
# 2) Mount Drive and reuse the V2 raw EEG cache and NeuroLM-B checkpoint.
from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis")
CACHE_ROOT = THESIS_ROOT / "CachedArtifacts/eeg_tokenizer/neurolm"
RESULTS_ROOT = THESIS_ROOT / "Results/eeg_tokenizer/neurolm"
RAW_PACKS = CACHE_ROOT / "raw_eeg_packs_v2"
CHECKPOINT_ROOT = CACHE_ROOT / "upstream_checkpoints"
RESULTS_DIR = RESULTS_ROOT / "partial_finetune_v5"
V4_RESULTS_DIR = RESULTS_ROOT / "gpt2_verbalizer_v4"

if not (RAW_PACKS / "cache_manifest.json").exists():
    raise FileNotFoundError("V2 raw packs are missing; finish V2's cache cell first")
for path in (CHECKPOINT_ROOT, RESULTS_DIR):
    path.mkdir(parents=True, exist_ok=True)
print("V2 raw packs:", RAW_PACKS)
print("V5 results:", RESULTS_DIR)

In [ ]:
# 3) Load the data, audited montage, and one reusable partially trainable model.
import json
import torch
from huggingface_hub import hf_hub_download
from src.channels import build_mne_spatial_mapping, select_usable_mapping
from src.config import CHECKPOINT_FILENAME, CHECKPOINT_REPOSITORY, PreprocessConfig
from src.gpt2_cache import OfficialNeuroLMGPT2
from src.partial_finetune import PartiallyUnfrozenNeuroLM, PartialFinetuneConfig
from src.raw_cache import load_raw_records

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → GPU, then rerun")
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
config = PartialFinetuneConfig()
records, recording_metadata, dataset_report = load_raw_records(RAW_PACKS, PreprocessConfig())
recording_metadata.to_csv(RESULTS_DIR / "recording_metadata.csv", index=False)
(RESULTS_DIR / "dataset_diagnostics.json").write_text(json.dumps(dataset_report, indent=2))

mapping_all, mapping = select_usable_mapping(build_mne_spatial_mapping())
mapping_all.to_csv(RESULTS_DIR / "spatial_mapping.csv", index=False)
mapping_report = {
    "assignments_total": len(mapping_all),
    "assignments_used": len(mapping),
    "assignments_excluded_over_30_deg": int((~mapping_all.use_for_encoder).sum()),
    "used_mean_angular_distance_deg": float(mapping.angular_distance_deg.mean()),
    "used_max_angular_distance_deg": float(mapping.angular_distance_deg.max()),
}
(RESULTS_DIR / "spatial_mapping_diagnostics.json").write_text(json.dumps(mapping_report, indent=2))

CACHED_CHECKPOINT = CHECKPOINT_ROOT / CHECKPOINT_FILENAME
if CACHED_CHECKPOINT.exists():
    CHECKPOINT_PATH = CACHED_CHECKPOINT
    print("Reusing the checkpoint already in Drive; no Hugging Face request is made.")
else:
    print("Checkpoint is absent; downloading the public official file once to Drive.")
    CHECKPOINT_PATH = Path(hf_hub_download(
        repo_id=CHECKPOINT_REPOSITORY, filename=CHECKPOINT_FILENAME, local_dir=CHECKPOINT_ROOT
    ))
checkpoint_bytes = CHECKPOINT_PATH.stat().st_size
if checkpoint_bytes < 2_000_000_000:
    raise IOError("Checkpoint is unexpectedly small or incomplete")
checkpoint_fingerprint = f"{CHECKPOINT_PATH.name}:{checkpoint_bytes}:{CHECKPOINT_PATH.stat().st_mtime_ns}"
base = OfficialNeuroLMGPT2(
    UPSTREAM_ROOT, CHECKPOINT_PATH, mapping.neurolm_index.to_numpy(),
    mapping.zuco_index.to_numpy(), device="cuda", maximum_seconds=config.maximum_seconds
)
model = PartiallyUnfrozenNeuroLM(base, config)
provenance = {
    "project_revision": project_revision,
    "upstream_commit": UPSTREAM_COMMIT,
    "checkpoint_repository": CHECKPOINT_REPOSITORY,
    "checkpoint_filename": CHECKPOINT_FILENAME,
    "checkpoint_bytes": checkpoint_bytes,
    "full_model_load": base.load_report,
    "trainability": model.report,
}
(RESULTS_DIR / "checkpoint_provenance.json").write_text(json.dumps(provenance, indent=2))
del base
print(json.dumps(dataset_report, indent=2))
print(json.dumps(mapping_report, indent=2))
print(json.dumps(model.report, indent=2))

In [ ]:
# 4) Verify one end-to-end forward/backward pass and the freeze boundary.
from src.partial_finetune import smoke_test_partial_finetune

smoke = smoke_test_partial_finetune(model, records, config)
print(json.dumps(smoke, indent=2))
if smoke["trainable_tokenizer_parameters"] != 0 or smoke["frozen_gradient_tensor_count"] != 0:
    raise RuntimeError("The V5 frozen/trainable boundary is incorrect")
print("Gradient check passed: tokenizer/lower layers frozen; selected top layers trainable.")

In [ ]:
# 5) Run/resume 3 seeds × 5 folds × aligned/shuffled fine-tuning.
from src.partial_finetune import evaluate_partial_finetune

metrics, predictions, summary, delta, gate = evaluate_partial_finetune(
    model=model,
    records=records,
    output_dir=RESULTS_DIR,
    dataset_fingerprint=dataset_report["dataset_fingerprint"],
    checkpoint_fingerprint=checkpoint_fingerprint,
    config=config,
)
display(summary)
print("Paired bootstrap:", json.dumps(delta, indent=2))
print("Decision:", gate["decision"])

In [ ]:
# 6) Show the clean seed-level result and, when available, the frozen V4 reference.
import matplotlib.pyplot as plt
import pandas as pd

metrics = pd.read_csv(RESULTS_DIR / "fold_metrics.csv")
gate = json.loads((RESULTS_DIR / "viability_gate.json").read_text())
seed_scores = metrics.groupby(["seed", "setup"])["macro_f1"].mean().unstack("setup")
display(seed_scores)
axes = seed_scores[["neurolm_partial_finetune", "neurolm_partial_finetune_shuffled", "majority"]].plot.bar(figsize=(8, 4))
axes.axhline(1 / 3, color="black", linestyle="--", linewidth=1, label="1/3 reference")
axes.set(title="V5 macro-F1 by split seed", ylabel="macro-F1", xlabel="seed")
axes.legend(loc="best")
plt.tight_layout()
figure_path = RESULTS_DIR / "macro_f1_by_seed.png"
plt.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()

v4_metrics_path = V4_RESULTS_DIR / "fold_metrics.csv"
if v4_metrics_path.exists():
    v4 = pd.read_csv(v4_metrics_path)
    comparison = pd.DataFrame({
        "V4 frozen aligned": [v4.loc[v4.setup == "neurolm_gpt2_verbalizer", "macro_f1"].mean()],
        "V5 partial aligned": [metrics.loc[metrics.setup == "neurolm_partial_finetune", "macro_f1"].mean()],
        "V5 partial shuffled": [metrics.loc[metrics.setup == "neurolm_partial_finetune_shuffled", "macro_f1"].mean()],
    }, index=["macro-F1"]).T
    display(comparison)
else:
    print("V4 fold metrics were not found; V5 results are still complete on their own.")
print(json.dumps(gate, indent=2))
print("Saved figure:", figure_path)

## Interpretation boundary

The primary question is whether **aligned V5 beats its independently trained shuffled-EEG control consistently**. A higher score than frozen V4 is interesting but not sufficient by itself, because V5 uses resource-bounded reader resampling during training. This notebook holds out sentences, not subjects; a green result earns a separately locked subject-and-sentence-independent confirmation.